In [ ]:
import f1_env
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

In [ ]:
class mlp_2hidden(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.ff1 = nn.Linear(input_dim, hidden_dim)
        self.ff2 = nn.Linear(hidden_dim, hidden_dim)
        self.ff3 = nn.Linear(hidden_dim, output_dim)

        nn.init.uniform_(self.ff1.weight, -0.001, 0.001)
        nn.init.uniform_(self.ff2.weight, -0.001, 0.001)
        nn.init.uniform_(self.ff3.weight, -0.001, 0.001)

        nn.init.uniform_(self.ff1.bias, -0.001, 0.001)
        nn.init.uniform_(self.ff2.bias, -0.001, 0.001)
        nn.init.uniform_(self.ff3.bias, -0.001, 0.001)

        self.ReLU = nn.ReLU()
    
    def forward(self, x):
        y = self.ReLU(self.ff1(x))
        y = self.ReLU(self.ff2(y))
        return self.ff3(y)

In [ ]:
def boltzmann(logits, T):
    prob = nn.Softmax(dim=-1)(logits/T)
    distribution = torch.distributions.Categorical(prob)
    action = distribution.sample()
    log_prob = distribution.log_prob(action)
    return action, log_prob

In [ ]:
GAMMA = 0.99

In [ ]:
class ActorCriticAgent:
    def __init__(self, lr_actor=1e-4, lr_critic=1e-3, input_length=8): # TBD
        self.actor = mlp_2hidden(input_length, 128, 4) 
        self.critic = mlp_2hidden(input_length, 128, 1) 
        self.optimizer_actor = torch.optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.optimizer_critic = torch.optim.Adam(self.critic.parameters(), lr=lr_critic)
        
    def select_action(self, state, T, action_mask): 
        state_tensor = torch.FloatTensor(state)
        logits = self.actor(state_tensor)

        action_mask = torch.FloatTensor(action_mask)

        logits = logits + (action_mask - 1) * 1e9

        action, log_prob = boltzmann(logits, T)
        return action, log_prob

    def update(self, log_prob, reward, state, next_state, done):

        state_tensor = torch.FloatTensor(state)
        Vs_t = self.critic(state_tensor)
        
        Vs_t1 = torch.tensor(0)
        if not done:
                next_state_tensor = torch.FloatTensor(next_state)
                Vs_t1 = self.critic(next_state_tensor)

        
        loss_actor = - (reward + GAMMA * Vs_t1.detach() - Vs_t.detach()) * log_prob
        loss_critic = (reward + GAMMA * Vs_t1.detach() - Vs_t)**2 
        
        
        self.optimizer_actor.zero_grad()
        loss_actor.backward()
        self.optimizer_actor.step()
        
        self.optimizer_critic.zero_grad()
        loss_critic.backward()
        self.optimizer_critic.step()

In [ ]:
EPISODES = 100

In [ ]:
def actorCritic(T = 10.0, lr_actor=1e-4, lr_critic=1e-3, input_length):
    agent = ActorCriticAgent(lr_actor, lr_critic, input_length=input_length)
    circuit_envs = _ #TBD
    all_returns =[]

    for _ in range(EPISODES):

        for circuit_name, yearly_envs in circuit_envs.items():
            for env in yearly_envs:
                state, _ = env.reset()
                used_compounds = [False, False, False]
                discounted_return = 0
                t = 0
                done = False
                while not done:
                    remaining_laps = _ #TBD state[x]

                    # Annealing
                    T_lap = max(0.5, T*(remaining_laps/env.total_laps))


                    action, log_prob = agent.select_action(state, T, used_compounds)

                    next_state, reward, done, _ = env.step(action.item())
                    
                    reward = _ #REWARD FUNCTION
                    
                    agent.update(log_prob, reward, state, next_state, done)
                    state = next_state
                    discounted_return += (GAMMA**t)*reward
                    t += 1


                all_returns.append(discounted_return)

        T = max(0.5, 0.99 * T)
    
    return all_returns

In [ ]:
years = [2023, 2024]
gp_list = ['Australian Grand Prix', 'Monaco Grand Prix', 'Monza']

def train_actor_critic(T=10.0, years=years, gp_list=gp_list):
    agent = ActorCriticAgent(input_length=8)
    all_returns = {gp: {year: [] for year in years} for gp in gp_list}

    for year in years:
        for gp in gp_list:
            env = f1_env.F1Env(year, gp, agent_type='actor_critic')
            for _ in range(EPISODES):
                state, _ = env.reset()
                t = 0
                done = False

                while not done:
                    action_mask = state['action_mask']
                    remaining_laps = state['state'][0]

                    T_lap = max(0.5, T*(remaining_laps/env.total_laps))

                    action, log_prob = agent.select_action(state['state'], T_lap, action_mask)
                    next_state, reward, done, _ = env.step(action.item())

                    agent.update(log_prob, reward, state, next_state, done)
                    state = next_state
                    t += 1

                all_returns[gp][year].append(env.total_race_time.total_seconds())
                T = max(0.5, 0.99 * T)

    return all_returns